GA4 DuckDB Query Script
=======================

Queries the GA4 JSONL export without loading the full file into memory.
DuckDB streams directly from the JSONL using read_json() with
format='newline_delimited' — only the rows needed per query are pulled.
 
For heavier repeated analysis, run Option 1 first to convert to Parquet,
then switch JSONL_PATH to the Parquet file in subsequent runs.
 
Usage:
    python3 ga4_duckdb.py
 
Requirements:
    pip install duckdb


In [47]:
import duckdb

JSONL_PATH   = "C:/Users/HP/OneDrive/Desktop/GA4 Data Generator/outputs/ga4_events.jsonl"   # path to JSONL file
PARQUET_PATH = "C:/Users/HP/OneDrive/Desktop/GA4 Data Generator/outputs/ga4_events.parquet" # written by Option 1 (convert to Parquet)
 
# DuckDB memory cap — keeps resident memory bounded even on large scans
MEMORY_LIMIT = "1GB"

In [48]:
con = duckdb.connect()
con.execute(f"SET memory_limit='{MEMORY_LIMIT}'")
con.execute("SET threads=4")

In [49]:
# =============================================================================
# CORE VIEW — streaming directly from JSONL, no full load
# =============================================================================
# DuckDB's read_json with format='newline_delimited' scans the file
# line-by-line and pushes down filters before materialising rows.
# Queries that filter on event_date or event_name touch only the
# matching rows in memory at any given time.
# =============================================================================
 
JSONL_SOURCE = f"""
    read_json(
        '{JSONL_PATH}',
        format        = 'newline_delimited',
        maximum_object_size = 10_000_000
    )
"""

In [50]:
# Convenience macro: extract a string event_param by key
# Usage: ep_str(event_params, 'page_location')
con.execute("""
    CREATE OR REPLACE MACRO ep_str(params, k) AS (
        (SELECT p.value.string_value
         FROM   (SELECT unnest(params) AS p)
         WHERE  p.key = k
         LIMIT  1)
    )
""")
 
con.execute("""
    CREATE OR REPLACE MACRO ep_int(params, k) AS (
        (SELECT p.value.int_value
         FROM   (SELECT unnest(params) AS p)
         WHERE  p.key = k
         LIMIT  1)
    )
""")


In [45]:
# =============================================================================
# HELPER
# =============================================================================
 
def run(label, sql):
    print(f"\n{'='*65}")
    print(f"  {label}")
    print('='*65)
    result = con.execute(sql).df()
    #print(result.to_string(index=False))
    return result

In [10]:
# =============================================================================
# 1. DAILY EVENT VOLUME
#    Sanity check — spot the session_start gap (Anomaly C) and any
#    traffic spikes or drops across the 90-day window.
# =============================================================================
 
run("Daily event volume by event_name", f"""
    SELECT
        event_date,
        COUNT(*)                                             AS total_events,
        COUNT(*) FILTER (WHERE event_name = 'session_start') AS session_starts,
        COUNT(*) FILTER (WHERE event_name = 'page_view')     AS page_views,
        COUNT(*) FILTER (WHERE event_name = 'purchase')      AS purchases
    FROM {JSONL_SOURCE}
    GROUP BY event_date
    ORDER BY event_date
""")


  Daily event volume by event_name


,event_date,total_events,session_starts,page_views,purchases
0,20241001,3710,770,2133,7
1,20241002,4795,1015,2746,11
2,20241003,5591,1184,3243,8
3,20241004,6481,1359,3762,11
4,20241005,4153,872,2364,9
...,...,...,...,...,...
87,20241227,4062,853,2354,6
88,20241228,3651,781,2082,6
89,20241229,3191,668,1853,6
90,20241230,5506,1153,3173,12


In [11]:
# =============================================================================
# 2. SESSION_START RATIO BY DATE
#    Detects Anomaly C: ratio drops to 0 when session_start events are
#    missing while page_views continue to fire normally.
# =============================================================================
 
run("session_start / page_view ratio by date (spot Anomaly C)", f"""
    SELECT
        event_date,
        COUNT(*) FILTER (WHERE event_name = 'session_start') AS session_starts,
        COUNT(*) FILTER (WHERE event_name = 'page_view')     AS page_views,
        ROUND(
            COUNT(*) FILTER (WHERE event_name = 'session_start') * 1.0
            / NULLIF(COUNT(*) FILTER (WHERE event_name = 'page_view'), 0),
        3) AS ss_pv_ratio
    FROM {JSONL_SOURCE}
    GROUP BY event_date
    ORDER BY event_date
""")


  session_start / page_view ratio by date (spot Anomaly C)


,event_date,session_starts,page_views,ss_pv_ratio
0,20241001,770,2133,0.361
1,20241002,1015,2746,0.370
2,20241003,1184,3243,0.365
3,20241004,1359,3762,0.361
4,20241005,872,2364,0.369
...,...,...,...,...
87,20241227,853,2354,0.362
88,20241228,781,2082,0.375
89,20241229,668,1853,0.360
90,20241230,1153,3173,0.363


In [13]:
# =============================================================================
# 3. PII DETECTION IN PAGE_LOCATION
#    Detects Anomaly A: scans page_location event_params for common
#    PII query parameter patterns (email, phone, name).
#    Returns the date, user_pseudo_id, and the offending URL.
# =============================================================================
 
run("PII leakage in page_location (Anomaly A)", f"""
    SELECT
        event_date,
        user_pseudo_id,
        ep_str(event_params, 'page_location') AS page_location
    FROM {JSONL_SOURCE}
    WHERE event_name IN ('page_view', 'session_start')
      AND (
            ep_str(event_params, 'page_location') LIKE '%email=%'
         --OR ep_str(event_params, 'page_location') LIKE '%user_email=%'
         --OR ep_str(event_params, 'page_location') LIKE '%customer_email=%'
         --OR ep_str(event_params, 'page_location') LIKE '%phone=%'
         --OR ep_str(event_params, 'page_location') LIKE '%tel=%'
         --OR ep_str(event_params, 'page_location') LIKE '%first_name=%'
      )
    ORDER BY event_date, user_pseudo_id
    LIMIT 50
""")



  PII leakage in page_location (Anomaly A)


,event_date,user_pseudo_id,page_location
0,20241022,1c529333-8b99-49a5-8949-2e2e8f859886,https://www.nordhaus-living.com/collections/ne...
1,20241022,363975f4-9f5c-4e2c-971f-e74af9602ea5,https://www.nordhaus-living.com/?email=william...
2,20241022,47dd7b13-e796-4e78-9b7e-69b8b22434bd,https://www.nordhaus-living.com/collections/sa...
3,20241022,51816c7d-a3af-4b57-89b7-9d3173d722e4,https://www.nordhaus-living.com/?email=henryni...
4,20241022,5801476c-df3b-4f29-abad-c0ee5a056cf3,https://www.nordhaus-living.com/collections/so...
5,20241022,5b04e458-fe8f-4961-a1da-c8dbe088fa09,https://www.nordhaus-living.com/collections/sa...
6,20241022,8cb575ba-c145-4640-a621-9ea3e7ed0dc9,https://www.nordhaus-living.com/collections/ch...
7,20241022,8d79b263-99da-4375-adbc-53d069e40f87,https://www.nordhaus-living.com/collections/ch...
8,20241022,a3055a10-1065-442a-a680-ecd43cc85f3a,https://www.nordhaus-living.com/collections/sa...
9,20241022,b20230ef-fac0-4b9c-9c93-45171f5e845e,https://www.nordhaus-living.com/collections/sa...


In [14]:
# =============================================================================
# 4. PII LEAKAGE SUMMARY BY DATE
#    Aggregate view of how many events per day carry PII in their URL —
#    useful for pinpointing the exact deploy window.
# =============================================================================
 
run("PII leakage count by date (Anomaly A)", f"""
    SELECT
        event_date,
        COUNT(*) AS pii_events
    FROM {JSONL_SOURCE}
    WHERE event_name IN ('page_view', 'session_start')
      AND (
            ep_str(event_params, 'page_location') LIKE '%email=%'
         OR ep_str(event_params, 'page_location') LIKE '%user_email=%'
         OR ep_str(event_params, 'page_location') LIKE '%phone=%'
         OR ep_str(event_params, 'page_location') LIKE '%first_name=%'
      )
    GROUP BY event_date
    ORDER BY event_date
""")



  PII leakage count by date (Anomaly A)


,event_date,pii_events
0,20241022,33
1,20241023,28
2,20241024,18
3,20241025,36
4,20241026,16
5,20241027,21


In [15]:
# =============================================================================
# 5. CROSS-DOMAIN TRACKING FAILURE
#    Detects Anomaly B: sessions where the first event is a page_view
#    on /order-confirmation with page_referrer from the payment gateway,
#    but no session_start and no attribution data.
# =============================================================================
 
run("Cross-domain sessions from payment gateway (Anomaly B)", f"""
    SELECT
        event_date,
        user_pseudo_id,
        ep_str(event_params, 'page_location')  AS page_location,
        ep_str(event_params, 'page_referrer')  AS page_referrer,
        session_traffic_source_last_click
    FROM {JSONL_SOURCE}
    WHERE event_name = 'page_view'
      AND ep_str(event_params, 'page_referrer') LIKE '%pay.stripe.com%'
    ORDER BY event_date
    LIMIT 30
""")
 


  Cross-domain sessions from payment gateway (Anomaly B)


,event_date,user_pseudo_id,page_location,page_referrer,session_traffic_source_last_click
0,20241105,0f86f408-d758-409d-8e2c-6e00f02f77f0,https://www.nordhaus-living.com/order-confirma...,https://pay.stripe.com/checkout,<NA>
1,20241105,42825b5e-7c0d-4318-8871-fc1a5a929d93,https://www.nordhaus-living.com/order-confirma...,https://pay.stripe.com/checkout,<NA>
2,20241105,d37f444e-cd75-416f-b0fe-a0c29d4298ac,https://www.nordhaus-living.com/order-confirma...,https://pay.stripe.com/checkout,<NA>
3,20241105,0de85542-aef9-4907-957c-1125252e8c93,https://www.nordhaus-living.com/order-confirma...,https://pay.stripe.com/checkout,<NA>
4,20241105,4cd23a0a-a960-445a-a0b1-72139b45278c,https://www.nordhaus-living.com/order-confirma...,https://pay.stripe.com/checkout,<NA>
5,20241105,bdfc3e97-8ade-4aa1-8f13-50e68edddbe3,https://www.nordhaus-living.com/order-confirma...,https://pay.stripe.com/checkout,<NA>
6,20241105,2368b668-80d5-466e-9252-436e2f9ebc4c,https://www.nordhaus-living.com/order-confirma...,https://pay.stripe.com/checkout,<NA>
7,20241105,a0925c89-6a47-48e6-bd95-7c2b323fc7ab,https://www.nordhaus-living.com/order-confirma...,https://pay.stripe.com/checkout,<NA>
8,20241105,33821415-cda2-41ac-86ed-891da97df229,https://www.nordhaus-living.com/order-confirma...,https://pay.stripe.com/checkout,<NA>
9,20241105,e0a9e5e0-88a8-44b5-9fca-e56c78c859e6,https://www.nordhaus-living.com/order-confirma...,https://pay.stripe.com/checkout,<NA>


In [16]:
# =============================================================================
# 6. CROSS-DOMAIN DAILY COUNT
#    How many sessions per day arrive from the payment gateway with broken
#    attribution — useful to size the impact window.
# =============================================================================
 
run("Cross-domain session count by date (Anomaly B)", f"""
    SELECT
        event_date,
        COUNT(DISTINCT user_pseudo_id) AS affected_users,
        COUNT(*)                       AS affected_events
    FROM {JSONL_SOURCE}
    WHERE ep_str(event_params, 'page_referrer') LIKE '%pay.stripe.com%'
    GROUP BY event_date
    ORDER BY event_date
""")
 


  Cross-domain session count by date (Anomaly B)


,event_date,affected_users,affected_events
0,20241105,166,171
1,20241106,156,158
2,20241107,149,149
3,20241108,142,142
4,20241109,102,102
5,20241110,88,88
6,20241111,76,76
7,20241112,157,161
8,20241113,128,129
9,20241114,82,83


In [17]:
# =============================================================================
# 7. MISSING ATTRIBUTION — (not set) SESSIONS
#    Detects Anomaly D: sessions where both collected_traffic_source and
#    session_traffic_source_last_click are NULL.
#    These appear as (not set) source/medium in attribution reports.
# =============================================================================
 
run("Missing attribution by date (Anomaly D)", f"""
    SELECT
        event_date,
        COUNT(DISTINCT user_pseudo_id) AS affected_users,
        COUNT(*)                       AS affected_events,
        ROUND(
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY event_date),
        1) AS pct_of_day
    FROM {JSONL_SOURCE}
    WHERE collected_traffic_source       IS NULL
      AND session_traffic_source_last_click IS NULL
      AND event_name != 'session_start'
    GROUP BY event_date
    ORDER BY event_date
""")



  Missing attribution by date (Anomaly D)


,event_date,affected_users,affected_events,pct_of_day
0,20241001,111,438,100.0
1,20241002,144,541,100.0
2,20241003,166,636,100.0
3,20241004,193,750,100.0
4,20241005,128,509,100.0
...,...,...,...,...
86,20241226,155,595,100.0
87,20241227,136,536,100.0
88,20241228,129,483,100.0
89,20241229,85,309,100.0


In [20]:
run("Revenue from (not set) sessions by date (Anomaly D)", f"""
    SELECT
        event_date,
        CASE
            WHEN collected_traffic_source IS NULL
             AND session_traffic_source_last_click IS NULL THEN '(not set)'
            ELSE COALESCE(
                    collected_traffic_source.manual_source,
                    traffic_source.source,
                    '(direct)'
                 )
        END                                           AS source,
        COUNT(*)                                      AS purchases,
        ROUND(SUM(ecommerce.purchase_revenue), 2)     AS revenue_eur
    FROM {JSONL_SOURCE}
    WHERE event_name = 'purchase'
      AND ecommerce IS NOT NULL
    GROUP BY event_date, source
    ORDER BY event_date, revenue_eur DESC
""")



  Revenue from (not set) sessions by date (Anomaly D)


,event_date,source,purchases,revenue_eur
0,20241001,google,6,2554.0
1,20241001,bing,1,229.0
2,20241002,google,10,5408.0
3,20241002,(not set),1,449.0
4,20241003,instagram.com,4,2476.0
...,...,...,...,...
288,20241229,google,6,5813.0
289,20241230,google,6,3524.0
290,20241230,(not set),4,3105.0
291,20241230,newsletter,1,1099.0


In [21]:
run("Overall funnel conversion rates", f"""
    SELECT
        COUNT(*) FILTER (WHERE event_name = 'session_start') AS sessions,
        COUNT(*) FILTER (WHERE event_name = 'view_item')     AS view_item,
        COUNT(*) FILTER (WHERE event_name = 'add_to_cart')   AS add_to_cart,
        COUNT(*) FILTER (WHERE event_name = 'begin_checkout')AS begin_checkout,
        COUNT(*) FILTER (WHERE event_name = 'purchase')      AS purchases,
        ROUND(
            COUNT(*) FILTER (WHERE event_name = 'purchase') * 100.0
            / NULLIF(COUNT(*) FILTER (WHERE event_name = 'session_start'), 0),
        2) AS session_to_purchase_pct
    FROM {JSONL_SOURCE}
""")


  Overall funnel conversion rates


,sessions,view_item,add_to_cart,begin_checkout,purchases,session_to_purchase_pct
0,77821,24441,3975,1980,697,0.9


In [23]:
run("Sessions by device category", f"""
    SELECT
        device.category         AS device_category,
        COUNT(*)                AS session_starts
    FROM {JSONL_SOURCE}
    WHERE event_name = 'session_start'
    GROUP BY device.category
    ORDER BY session_starts DESC
""")



  Sessions by device category


,device_category,session_starts
0,desktop,45985
1,mobile,30331
2,tablet,1505


In [24]:
 
run("Top 10 countries by session volume", f"""
    SELECT
        geo.country             AS country,
        COUNT(*)                AS session_starts
    FROM {JSONL_SOURCE}
    WHERE event_name = 'session_start'
    GROUP BY geo.country
    ORDER BY session_starts DESC
    LIMIT 10
""")



  Top 10 countries by session volume


,country,session_starts
0,Germany,51837
1,Austria,6934
2,Switzerland,5422
3,Netherlands,4512
4,United Kingdom,3694
5,France,3206
6,Sweden,2216


In [25]:
run("Top products by revenue", f"""
    SELECT
        item.item_name,
        item.item_category,
        SUM(item.quantity)          AS units_sold,
        ROUND(SUM(item.item_revenue), 2) AS revenue_eur
    FROM {JSONL_SOURCE},
         LATERAL unnest(items) AS t(item)
    WHERE event_name = 'purchase'
    GROUP BY item.item_name, item.item_category
    ORDER BY revenue_eur DESC
    LIMIT 10
""")



  Top products by revenue


,item_name,item_category,units_sold,revenue_eur
0,Oslo Corner Sofa,Furniture,79.0,102621.0
1,Voss King Bed Frame,Furniture,81.0,89019.0
2,Bergen 3-Seater,Furniture,72.0,61128.0
3,Alesund Storage Bed,Furniture,69.0,55131.0
4,Stavanger Oak Desk,Furniture,57.0,39843.0
5,Tana Wool Rug 200x300,Textiles,70.0,31430.0
6,Fjord Accent Chair,Furniture,56.0,22344.0
7,Lofoten Coffee Table,Furniture,62.0,20398.0
8,Narvik Floor Lamp,Lighting,65.0,14885.0
9,Kirkenes Pendant Lamp,Lighting,77.0,11473.0


In [53]:
sql = f"""
WITH session_base AS (
    SELECT
        user_pseudo_id,
        ep_int(event_params, 'ga_session_id')                          AS ga_session_id,
        MIN(event_timestamp)                                           AS session_first_event_ts,
        MAX(event_timestamp)                                           AS session_last_event_ts,
        COUNT(*) FILTER (WHERE event_name = 'session_start')           AS has_session_start,

        -- User first-touch (immutable — not useful for E detection)
        FIRST(traffic_source.source ORDER BY event_timestamp)          AS first_touch_source,

        -- Collected source: event-level UTM, wiped to NULL for E sessions
        FIRST(collected_traffic_source.manual_source  ORDER BY event_timestamp) AS collected_source,
        FIRST(collected_traffic_source.manual_medium  ORDER BY event_timestamp) AS collected_medium,
        FIRST(collected_traffic_source.manual_campaign_name ORDER BY event_timestamp) AS collected_campaign,

        FIRST(ep_str(event_params, 'page_location') ORDER BY event_timestamp ASC)  AS first_page_location,
        FIRST(ep_str(event_params, 'page_location') ORDER BY event_timestamp DESC) AS last_page_location,

        CAST(
            timezone('Europe/Berlin', make_timestamp(MIN(event_timestamp)))
            AS DATE
        ) AS session_date

    FROM {JSONL_SOURCE}
    GROUP BY user_pseudo_id, ga_session_id
),
session_with_rank AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_pseudo_id
            ORDER BY session_first_event_ts ASC
        ) AS session_rn
    FROM session_base
),
session_with_prev AS (
    SELECT
        curr.*,
        prev.ga_session_id             AS prev_session_id,
        prev.collected_source          AS prev_collected_source,
        prev.collected_medium          AS prev_collected_medium,
        prev.collected_campaign        AS prev_collected_campaign,
        prev.last_page_location        AS prev_last_page_location,
        prev.session_last_event_ts     AS prev_session_last_event_ts,
        prev.session_date              AS prev_session_date,
        prev.has_session_start         AS prev_has_session_start
    FROM session_with_rank curr
    LEFT JOIN session_with_rank prev
        ON  curr.user_pseudo_id = prev.user_pseudo_id
        AND curr.session_rn     = prev.session_rn + 1
)
SELECT
    user_pseudo_id,
    ga_session_id,
    session_date,
    session_first_event_ts,

    -- These will be NULL for Anomaly E sessions
    collected_source,
    collected_medium,
    collected_campaign,

    -- First-touch shown for contrast — will still show google/organic etc.
    first_touch_source,

    has_session_start,
    first_page_location,

    -- Previous session context
    prev_session_id,
    prev_collected_source,
    prev_collected_medium,
    prev_session_date,
    prev_has_session_start,

    -- Gap from previous session in minutes
    ROUND(
        (session_first_event_ts - prev_session_last_event_ts) / 1e6 / 60,
        1
    ) AS gap_minutes_from_prev,

    (prev_last_page_location = first_page_location) AS same_page_continuation

FROM session_with_prev
WHERE
    has_session_start = 0
    AND collected_source IS NULL
    AND prev_session_last_event_ts IS NOT NULL
    AND same_page_continuation = 'True'

ORDER BY user_pseudo_id, session_first_event_ts
"""
result = con.execute(sql).df()
result

,user_pseudo_id,ga_session_id,session_date,session_first_event_ts,collected_source,collected_medium,collected_campaign,first_touch_source,has_session_start,first_page_location,prev_session_id,prev_collected_source,prev_collected_medium,prev_session_date,prev_has_session_start,gap_minutes_from_prev,same_page_continuation
0,001f65a9-5917-4218-b601-7f6df01e076c,4184032863,2025-11-20,1763679308000000,None,None,None,google,0,https://www.nordhaus-living.com/collections/ch...,9395655398,google,organic,2025-11-13,1,11080.8,True
1,003615d7-aa0d-4557-be3f-850f57bd805f,1736740398,2025-11-18,1763493205000000,None,None,None,google,0,https://www.nordhaus-living.com/,8092668738,None,None,2025-11-16,0,3084.4,True
2,00563b0c-18d4-4b8d-9b8b-2979505d838a,5152380255,2025-12-19,1766152231000000,None,None,None,(direct),0,https://www.nordhaus-living.com/collections/sofas,9975953013,google,organic,2025-11-26,1,32870.0,True
3,00755640-6d0c-4cce-bd0f-b50ad6385a69,1253745685,2025-11-09,1762663066000000,None,None,None,instagram.com,0,https://www.nordhaus-living.com/order-confirma...,6411577568,None,None,2025-11-08,0,1021.3,True
4,00b91332-fd97-4443-a6e5-b506b097de70,5007507379,2025-11-20,1763671771000000,None,None,None,newsletter,0,https://www.nordhaus-living.com/order-confirma...,1051080520,None,None,2025-11-17,0,4387.8,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
812,fe90e45c-0425-47bc-9211-b1cb025bbc23,7611199729,2025-12-19,1766118548000000,None,None,None,google,0,https://www.nordhaus-living.com/collections/sale,2663850255,None,None,2025-12-17,1,2643.7,True
813,fea66377-dd35-4d13-b379-c0d79ccd5743,3547187583,2025-11-14,1763117957000000,None,None,None,google,0,https://www.nordhaus-living.com/collections/ne...,5704423550,google,cpc,2025-11-12,1,2305.1,True
814,ff122e26-d433-4ae7-bdaa-c0393a5eee1d,5871060480,2025-12-25,1766664269000000,None,None,None,google,0,https://www.nordhaus-living.com/collections/sofas,3127579883,None,None,2025-12-20,1,7241.9,True
815,ff19cac0-4956-4bc6-8c04-ad6d52b47e17,9323376666,2025-11-26,1764145725000000,None,None,None,google,0,https://www.nordhaus-living.com/order-confirma...,8632497298,None,None,2025-11-24,1,2193.5,True


In [40]:
con.close()